# Locations vs Dimensions

In grid_py (as in R's grid), the same unit type can behave differently
depending on whether it represents a **location** or a **dimension**.

* A *location* is a position on an axis (e.g., "where to place a point").
* A *dimension* is a size (e.g., "how wide should this rectangle be").

The distinction matters most for `"native"` units, where the origin and
direction of the axis affect the conversion. This notebook demonstrates
the difference and shows how to use the conversion functions
`convert_x`, `convert_y`, `convert_width`, and `convert_height`.

The narrative follows the original R vignette (`locndimn.Rnw`).

In [ ]:
import matplotlib
matplotlib.use("Agg")

import grid_py as gp

## Native Units

The `"native"` unit type refers to the coordinate system defined by
a viewport's `xscale` and `yscale`. A value of `50` in native units
means different absolute lengths depending on the viewport's scale
and physical size.

In [ ]:
# A viewport with native x-range [0, 100] and y-range [0, 200]
vp = gp.Viewport(
    xscale=[0, 100],
    yscale=[0, 200],
    width=gp.Unit(4, "inches"),
    height=gp.Unit(4, "inches"),
    name="native_demo",
)

print("xscale:", vp.xscale)
print("yscale:", vp.yscale)

In [ ]:
# Draw a point at native location (50, 100)
gp.grid_newpage()
gp.push_viewport(vp)

gp.grid_rect(gp=gp.Gpar(col="grey", lty="dashed"))

gp.grid_points(
    x=gp.Unit(50, "native"),
    y=gp.Unit(100, "native"),
    gp=gp.Gpar(col="red"),
)
gp.grid_text(
    "(50, 100)",
    x=gp.Unit(50, "native"),
    y=gp.Unit(110, "native"),
    gp=gp.Gpar(fontsize=10),
)

gp.pop_viewport()

## Location Conversions: convert_x / convert_y

`convert_x` and `convert_y` convert a **location** from one unit type
to another. The conversion takes the viewport's scale and position
into account.

In [ ]:
gp.grid_newpage()
gp.push_viewport(vp)

# Convert a native x-location to NPC
loc_native = gp.Unit(50, "native")
loc_npc = gp.convert_x(loc_native, "npc")
print("native x=50 in npc:", loc_npc)

# Convert a native y-location to NPC
loc_y_native = gp.Unit(100, "native")
loc_y_npc = gp.convert_y(loc_y_native, "npc")
print("native y=100 in npc:", loc_y_npc)

gp.pop_viewport()

## Dimension Conversions: convert_width / convert_height

`convert_width` and `convert_height` convert a **size** rather than a
position. For `"native"` units the difference is crucial: a width of 50
native units spans half the axis range, regardless of where it starts.

In [ ]:
gp.grid_newpage()
gp.push_viewport(vp)

# Convert a native width to NPC
w_native = gp.Unit(50, "native")
w_npc = gp.convert_width(w_native, "npc")
print("native width=50 in npc:", w_npc)

# Convert a native height to NPC
h_native = gp.Unit(100, "native")
h_npc = gp.convert_height(h_native, "npc")
print("native height=100 in npc:", h_npc)

gp.pop_viewport()

## Comparing Location and Dimension

The key insight: `convert_x(Unit(50, "native"), "npc")` gives the
*position* 0.5 npc (the midpoint), while `convert_width(Unit(50, "native"), "npc")`
gives the *size* 0.5 npc (half the viewport width). In this simple
example they happen to agree because `xscale` starts at 0, but with a
non-zero origin they differ.

In [ ]:
# Viewport with an offset native range
vp_offset = gp.Viewport(
    xscale=[10, 110],
    yscale=[0, 200],
    width=gp.Unit(4, "inches"),
    height=gp.Unit(4, "inches"),
    name="offset_demo",
)

gp.grid_newpage()
gp.push_viewport(vp_offset)

# Location: native x = 60 is at the midpoint of [10, 110]
loc = gp.convert_x(gp.Unit(60, "native"), "npc")
print("location  native x=60 -> npc:", loc)

# Dimension: a width of 50 is half the range (100 units wide)
dim = gp.convert_width(gp.Unit(50, "native"), "npc")
print("dimension native w=50 -> npc:", dim)

gp.pop_viewport()

## The General convert_unit Function

`convert_unit` is the most general conversion function. It takes an
extra argument specifying whether the value is a location (`"x"`, `"y"`)
or a dimension (`"width"`, `"height"`).

In [ ]:
gp.grid_newpage()
gp.push_viewport(vp)

# Convert using the general function
val = gp.Unit(2, "cm")
print("2 cm as inches (dimension):", gp.convert_unit(val, "inches"))

gp.pop_viewport()

## Summary

* `"native"` units depend on the viewport's `xscale` / `yscale`.
* **Locations** (`convert_x`, `convert_y`) map a position on the axis.
* **Dimensions** (`convert_width`, `convert_height`) map a size.
* With a non-zero axis origin, converting a *location* and a *dimension*
  with the same numeric value yields different results.
* `convert_unit` is the general-purpose wrapper.